# Preprocessing Dataset: Keyphrase Generation Berbasis T5-Small
**Proyek Akhir - Pemrosesan Bahasa Alami (SINF6054)**  
Kelompok 3: Tasya Zahrani, Firah Maulida, Muhammad Reky

Notebook ini melakukan preprocessing dataset `hasil_indonesia_gabungan_clean.csv` mencakup:
- Penghapusan kolom kosong / missing value
- Deteksi dan filter bahasa Indonesia (multi-lapis)
- Pembersihan teks abstrak dan keyphrase
- Filtrasi panjang teks
- **Truncation token** (bukan hapus) agar dataset tidak berkurang drastis
- Pembagian dataset 80/10/10 dan penyimpanan

## 1. Instalasi dan Import Library

In [1]:
!pip install langdetect transformers sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 17.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import pandas as pd
import numpy as np
import re
import os
import json
import unicodedata
from tqdm import tqdm

from langdetect import detect_langs, DetectorFactory, LangDetectException
from sklearn.model_selection import train_test_split
from transformers import T5Tokenizer

# Hasil langdetect deterministik
DetectorFactory.seed = 42

# Aktifkan tqdm untuk pandas
tqdm.pandas()

print("Import selesai.")

Import selesai.


## 2. Muat Dataset

In [3]:
# Cari path file otomatis
print("File yang tersedia di /kaggle/input:")
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

File yang tersedia di /kaggle/input:
/kaggle/input/datasets/firahmaulida/dataset/dataset_id_3kolom.csv


In [4]:
# Path dataset (sudah dikonfirmasi dari output os.walk di atas)
CSV_PATH = "/kaggle/input/datasets/firahmaulida/dataset/dataset_id_3kolom.csv"

df = pd.read_csv(CSV_PATH, dtype=str)

print(f"Total baris awal  : {len(df)}")
print(f"Kolom yang ada    : {df.columns.tolist()}")
print("\nPreview 3 baris pertama:")
df.head(3)

Total baris awal  : 9461
Kolom yang ada    : ['judul', 'abstrak', 'kata_kunci']

Preview 3 baris pertama:


,judul,abstrak,kata_kunci
0,Analisis Hubungan Penguasaan Kosakata dan Kema...,This research problem is how is the relation b...,Penguasaan Kosakata; Unsur Intrinsik Cerpen; S...
1,Evaluasi Penggunaan Aplikasi Sistem Keuangan D...,Aplikasi Siskeudes ini merupakan suatu sistem ...,evaluasi; keuangan desa; siskeudes; sistem inf...
2,ANALISIS TEKNIS DAN FINANSIAL PRODUKSI ARANG D...,Limbah serbuk gergaji dan sebetan dari industr...,Serbuk gergaji; sebetan; arang; cuka kayu; tun...


In [5]:
# Nama kolom sesuai CSV (judul, abstrak, kata_kunci)
KOLOM_JUDUL     = "judul"
KOLOM_ABSTRAK   = "abstrak"
KOLOM_KEYPHRASE = "kata_kunci"

# Verifikasi kolom tersedia
for kol in [KOLOM_ABSTRAK, KOLOM_KEYPHRASE]:
    assert kol in df.columns, (
        f"Kolom '{kol}' tidak ditemukan. "
        f"Kolom yang ada: {df.columns.tolist()}"
    )

print("Kolom berhasil diverifikasi.")

Kolom berhasil diverifikasi.


## 3. Hapus Kolom Kosong dan Missing Value

In [6]:
jumlah_awal = len(df)

# 3a. Hapus kolom yang seluruhnya kosong
kolom_sebelum = df.columns.tolist()
df.dropna(axis=1, how="all", inplace=True)
kolom_dihapus = set(kolom_sebelum) - set(df.columns.tolist())
if kolom_dihapus:
    print(f"Kolom dihapus (seluruhnya kosong): {kolom_dihapus}")
else:
    print("Tidak ada kolom yang seluruhnya kosong.")

# 3b. Hapus kolom yang >80% nilainya kosong
ambang_kosong = 0.80
kolom_hampir_kosong = [
    kol for kol in df.columns
    if df[kol].isna().mean() > ambang_kosong
]
if kolom_hampir_kosong:
    df.drop(columns=kolom_hampir_kosong, inplace=True)
    print(f"Kolom dihapus (>80% kosong): {kolom_hampir_kosong}")
else:
    print("Tidak ada kolom dengan >80% nilai kosong.")

# 3c. Tampilkan missing value per kolom
print("\nMissing value per kolom:")
print(df.isnull().sum())

# 3d. Hapus baris di mana abstrak atau keyphrase kosong
df.dropna(subset=[KOLOM_ABSTRAK, KOLOM_KEYPHRASE], inplace=True)
df = df[df[KOLOM_ABSTRAK].str.strip().str.len() > 0]
df = df[df[KOLOM_KEYPHRASE].str.strip().str.len() > 0]
df.reset_index(drop=True, inplace=True)

print(f"\nBaris sebelum hapus missing value : {jumlah_awal}")
print(f"Baris setelah hapus missing value  : {len(df)}")
print(f"Baris dihapus                      : {jumlah_awal - len(df)}")

Tidak ada kolom yang seluruhnya kosong.
Tidak ada kolom dengan >80% nilai kosong.

Missing value per kolom:
judul         0
abstrak       0
kata_kunci    0
dtype: int64

Baris sebelum hapus missing value : 9461
Baris setelah hapus missing value  : 9461
Baris dihapus                      : 0


## 4. Deteksi Bahasa Indonesia (Multi-Lapis)

Empat sinyal digunakan:
- **S1** – `langdetect` pada kolom abstrak
- **S2** – Penanda bilingual: kata `ABSTRAK` di tengah teks
- **S3** – `langdetect` pada kolom keyphrase
- **S4** – Rasio stopword Indonesia sebagai fallback

Baris dianggap Indonesia jika **minimal satu sinyal** bernilai True.

In [7]:
# Konfigurasi deteksi bahasa
LANGDETECT_MIN_PROB       = 0.50
LANGDETECT_CHARS_ABSTRAK  = 700
LANGDETECT_CHARS_KATKUNCI = 300
STOPWORD_RATIO_THRESHOLD  = 0.08
BILINGUAL_PATTERN         = re.compile(r"\bABSTRAK\b", re.IGNORECASE)

# Stopword Indonesia yang distingtif
ID_STOPWORDS = frozenset({
    "dan", "yang", "dalam", "dengan", "untuk", "adalah", "bahwa", "pada",
    "tidak", "dari", "atau", "juga", "dapat", "telah", "akan", "penelitian",
    "hasil", "oleh", "sebagai", "tersebut", "lebih", "serta", "antara",
    "secara", "setelah", "dilakukan", "menunjukkan", "digunakan",
    "berdasarkan", "sehingga", "memiliki", "merupakan", "terdapat",
    "diperoleh", "mengenai", "penggunaan", "pengembangan", "pendidikan",
    "masyarakat", "terhadap", "peningkatan", "metode", "tujuan",
    "pengaruh", "analisis", "faktor", "kualitas", "diharapkan",
    "ditemukan", "menjelaskan", "diketahui", "penulis", "artikel",
    "jurnal", "studi", "kajian",
})


def langdetect_prob(teks: str, max_chars: int) -> dict:
    potongan = str(teks).strip()[:max_chars]
    if not potongan:
        return {}
    try:
        return {r.lang: r.prob for r in detect_langs(potongan)}
    except LangDetectException:
        return {}


def is_indonesian_langdetect(teks: str, max_chars: int) -> bool:
    probs = langdetect_prob(teks, max_chars)
    return probs.get("id", 0.0) >= LANGDETECT_MIN_PROB


def is_bilingual(teks: str) -> bool:
    return bool(BILINGUAL_PATTERN.search(str(teks)))


def stopword_ratio(teks: str) -> float:
    kata = re.findall(r"\b[a-zA-Z]+\b", str(teks).lower())
    if not kata:
        return 0.0
    return sum(1 for k in kata if k in ID_STOPWORDS) / len(kata)


def klasifikasi_bahasa(row) -> dict:
    abstrak   = str(row[KOLOM_ABSTRAK])
    keyphrase = str(row[KOLOM_KEYPHRASE])

    s1 = is_indonesian_langdetect(abstrak, LANGDETECT_CHARS_ABSTRAK)
    s2 = is_bilingual(abstrak)
    s3 = is_indonesian_langdetect(keyphrase, LANGDETECT_CHARS_KATKUNCI)
    s4_rasio = stopword_ratio(abstrak)
    s4 = s4_rasio >= STOPWORD_RATIO_THRESHOLD

    is_id = s1 or s2 or (s3 and s4)

    if is_id and s2 and not s1:
        kategori = "BILINGUAL (ID+EN)"
    elif is_id:
        kategori = "INDONESIA"
    else:
        kategori = "NON-INDONESIA"

    return {
        "s1_langdetect_abstrak" : s1,
        "s2_bilingual_marker"   : s2,
        "s3_langdetect_keyword" : s3,
        "s4_stopword_ratio"     : round(s4_rasio, 4),
        "is_indonesian"         : is_id,
        "kategori_bahasa"       : kategori,
    }


print("Mendeteksi bahasa setiap baris — harap tunggu...")
hasil_deteksi = df.progress_apply(klasifikasi_bahasa, axis=1, result_type="expand")
df = pd.concat([df, hasil_deteksi], axis=1)

print("\nDistribusi kategori bahasa:")
print(df["kategori_bahasa"].value_counts())

Mendeteksi bahasa setiap baris — harap tunggu...


100%|██████████| 9461/9461 [01:14<00:00, 127.52it/s]


Distribusi kategori bahasa:
kategori_bahasa
INDONESIA            7975
BILINGUAL (ID+EN)     947
NON-INDONESIA         539
Name: count, dtype: int64


In [8]:
# Filter: pertahankan hanya baris berbahasa Indonesia (termasuk bilingual)
sebelum_filter_bahasa = len(df)
df = df[df["is_indonesian"] == True].reset_index(drop=True)

print(f"Baris sebelum filter bahasa : {sebelum_filter_bahasa}")
print(f"Baris setelah filter bahasa : {len(df)}")
print(f"Baris non-Indonesia dihapus : {sebelum_filter_bahasa - len(df)}")

Baris sebelum filter bahasa : 9461
Baris setelah filter bahasa : 8922
Baris non-Indonesia dihapus : 539


## 4b. Hapus Baris Bilingual (Pertahankan Murni Indonesia)

Baris dengan kategori `BILINGUAL (ID+EN)` dihapus agar dataset **murni Bahasa Indonesia**,
sehingga model T5 tidak bingung dan output keyphrase lebih konsisten dalam satu bahasa.

In [9]:
# ─────────────────────────────────────────────────────────────
# HAPUS BARIS BILINGUAL — pertahankan hanya murni Indonesia
# ─────────────────────────────────────────────────────────────
sebelum_filter_bilingual = len(df)

df = df[df['kategori_bahasa'] == 'INDONESIA'].reset_index(drop=True)

print('Distribusi kategori bahasa SEBELUM filter bilingual:')
print(f"  INDONESIA          : {sebelum_filter_bilingual - df['kategori_bahasa'].value_counts().reindex(['BILINGUAL (ID+EN)'], fill_value=0).sum()}")
print(f"  BILINGUAL (ID+EN)  : {sebelum_filter_bilingual - len(df)}")
print()
print(f'Baris sebelum filter bilingual : {sebelum_filter_bilingual}')
print(f'Baris dihapus (bilingual)      : {sebelum_filter_bilingual - len(df)}')
print(f'Baris setelah filter bilingual : {len(df)}')
print()
print('Distribusi kategori bahasa SETELAH filter:')
print(df['kategori_bahasa'].value_counts().to_string())


Distribusi kategori bahasa SEBELUM filter bilingual:
  INDONESIA          : 8922
  BILINGUAL (ID+EN)  : 947

Baris sebelum filter bilingual : 8922
Baris dihapus (bilingual)      : 947
Baris setelah filter bilingual : 7975

Distribusi kategori bahasa SETELAH filter:
kategori_bahasa
INDONESIA    7975


## 5. Pembersihan Teks (Text Cleaning)

In [10]:
def bersihkan_abstrak(teks: str) -> str:
    """Bersihkan teks abstrak untuk input model T5."""
    if pd.isna(teks) or not isinstance(teks, str):
        return ""

    teks = unicodedata.normalize("NFC", teks)

    # Untuk abstrak bilingual: ambil bagian setelah kata 'ABSTRAK'
    match_bilingual = re.search(r"\bABSTRAK\b", teks, re.IGNORECASE)
    if match_bilingual:
        teks = teks[match_bilingual.start():]
        teks = re.sub(r"^ABSTRAK[\s:.-]*", "", teks, flags=re.IGNORECASE)

    # Hapus tag HTML
    teks = re.sub(r"<[^>]+>", " ", teks)

    # Ganti baris baru / tab jadi spasi
    teks = re.sub(r"[\r\n\t]+", " ", teks)

    # Hapus URL
    teks = re.sub(r"http\S+|www\.\S+", "", teks)

    # Pertahankan huruf, angka, spasi, tanda baca dasar
    teks = re.sub(r"[^\w\s.,;:()?!\-]", " ", teks)

    # Normalisasi spasi
    teks = re.sub(r"\s+", " ", teks).strip()

    # Lowercase
    teks = teks.lower()

    return teks


def bersihkan_keyphrase(teks: str) -> str:
    """Bersihkan kolom keyphrase menjadi daftar dipisah koma."""
    if pd.isna(teks) or not isinstance(teks, str):
        return ""

    teks = unicodedata.normalize("NFC", teks)

    # Normalisasi pemisah alternatif (titik koma, slash) menjadi koma
    teks = re.sub(r"[;|/]", ",", teks)

    # Pisah, trim, lowercase
    daftar = [k.strip().lower() for k in teks.split(",")]

    # Bersihkan karakter non-kata di awal/akhir
    daftar = [re.sub(r"^[^\w]+|[^\w]+$", "", k) for k in daftar]

    # Hapus yang terlalu pendek
    daftar = [k for k in daftar if len(k) >= 2]

    # Hapus duplikat, pertahankan urutan
    dilihat = set()
    hasil = []
    for k in daftar:
        if k not in dilihat:
            dilihat.add(k)
            hasil.append(k)

    return ", ".join(hasil)


print("Membersihkan kolom abstrak...")
df["abstrak_bersih"] = df[KOLOM_ABSTRAK].progress_apply(bersihkan_abstrak)

print("Membersihkan kolom keyphrase...")
df["keyphrase_bersih"] = df[KOLOM_KEYPHRASE].progress_apply(bersihkan_keyphrase)

print("Pembersihan selesai.")

Membersihkan kolom abstrak...


100%|██████████| 7975/7975 [00:01<00:00, 5060.84it/s]


Membersihkan kolom keyphrase...


100%|██████████| 7975/7975 [00:00<00:00, 90670.63it/s]

Pembersihan selesai.


## 6. Filtrasi Data (Hapus Baris Tidak Valid)

In [11]:
sebelum_filtrasi = len(df)

# Hapus abstrak atau keyphrase kosong setelah dibersihkan
df = df[df["abstrak_bersih"].str.len() > 0]
df = df[df["keyphrase_bersih"].str.len() > 0]

# Hapus abstrak terlalu pendek
# Catatan: batas atas dihapus karena token panjang akan di-truncate, bukan dihapus
PANJANG_MIN = 50
df = df[df["abstrak_bersih"].str.len() >= PANJANG_MIN]

# Hitung jumlah keyphrase bersih
df["jumlah_keyphrase_bersih"] = df["keyphrase_bersih"].apply(
    lambda x: len([k.strip() for k in x.split(",") if k.strip()])
)
df = df[df["jumlah_keyphrase_bersih"] >= 1]

# Hapus duplikat berdasarkan abstrak
df = df.drop_duplicates(subset=["abstrak_bersih"]).reset_index(drop=True)

print(f"Baris sebelum filtrasi : {sebelum_filtrasi}")
print(f"Baris setelah filtrasi : {len(df)}")
print(f"Baris dihapus          : {sebelum_filtrasi - len(df)}")

Baris sebelum filtrasi : 7975
Baris setelah filtrasi : 7969
Baris dihapus          : 6


## 7. Format Input T5 (Prefix `generate keyphrases:`)

In [12]:
T5_PREFIX = "generate keyphrases: "

df["input_model"]  = T5_PREFIX + df["abstrak_bersih"]
df["target_model"] = df["keyphrase_bersih"]

print("Contoh input_model:")
print(df["input_model"].iloc[0][:200], "...")
print("\nContoh target_model:")
print(df["target_model"].iloc[0])

Contoh input_model:
generate keyphrases: aplikasi siskeudes ini merupakan suatu sistem yang dikembangkan oleh bpkp dalam meningkatkan kualitas tata kelola keuangan desa. pemerintah padang pariaman sudah menetapkan siskeu ...

Contoh target_model:
evaluasi, keuangan desa, siskeudes, sistem informasi manajemen


## 8. Truncation Token (Bukan Hapus)

T5-small memiliki batas 512 token input. Sebelumnya notebook menghapus entri yang melebihi batas ini,
sehingga dataset menyusut dari **679 → 128 entri** (terlalu sedikit untuk fine-tuning).

Solusi: **truncation** — input yang lebih dari 512 token dipotong hingga tepat 512 token,
lalu di-decode kembali menjadi teks. Data tidak berkurang, hanya bagian akhir abstrak yang terpotong.
Ini aman karena inti informasi abstrak umumnya ada di bagian awal.

In [13]:
print("Memuat tokenizer T5-small...")
tokenizer = T5Tokenizer.from_pretrained("t5-small")

MAX_INPUT_TOKEN  = 512
MAX_TARGET_TOKEN = 64


def hitung_token(teks: str) -> int:
    """Hitung jumlah token tanpa truncation."""
    return len(tokenizer.encode(teks, truncation=False))


# Cek panjang token SEBELUM truncation
print("Menghitung panjang token sebelum truncation...")
df["token_input_raw"]  = df["input_model"].progress_apply(hitung_token)
df["token_target"]     = df["target_model"].progress_apply(hitung_token)

print("\nStatistik token INPUT (sebelum truncation):")
print(df["token_input_raw"].describe().round(1))
print(f"Entri melebihi {MAX_INPUT_TOKEN} token: {(df['token_input_raw'] > MAX_INPUT_TOKEN).sum()} dari {len(df)}")

print("\nStatistik token TARGET (keyphrase):")
print(df["token_target"].describe().round(1))
print(f"Entri melebihi {MAX_TARGET_TOKEN} token: {(df['token_target'] > MAX_TARGET_TOKEN).sum()}")

Memuat tokenizer T5-small...


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Menghitung panjang token sebelum truncation...


100%|██████████| 7969/7969 [00:00<00:00, 12245.21it/s]


Statistik token INPUT (sebelum truncation):
count    7969.0
mean      687.2
std       237.1
min        36.0
25%       536.0
50%       662.0
75%       803.0
max      3330.0
Name: token_input_raw, dtype: float64
Entri melebihi 512 token: 6284 dari 7969

Statistik token TARGET (keyphrase):
count    7969.0
mean       26.9
std        11.1
min         2.0
25%        20.0
50%        26.0
75%        33.0
max       312.0
Name: token_target, dtype: float64
Entri melebihi 64 token: 43


In [14]:
def truncate_input(teks: str) -> str:
    """
    Potong teks input ke maksimal MAX_INPUT_TOKEN token.
    Token di-decode kembali menjadi teks agar tetap bisa dibaca model.
    """
    token_ids = tokenizer.encode(teks, truncation=True, max_length=MAX_INPUT_TOKEN)
    return tokenizer.decode(token_ids, skip_special_tokens=True)


print(f"Melakukan truncation input ke {MAX_INPUT_TOKEN} token...")
df["input_model"] = df["input_model"].progress_apply(truncate_input)

# Hitung ulang token setelah truncation (verifikasi)
df["token_input"] = df["input_model"].progress_apply(hitung_token)

print("\nStatistik token INPUT (setelah truncation):")
print(df["token_input"].describe().round(1))
print(f"Entri masih melebihi {MAX_INPUT_TOKEN} token: {(df['token_input'] > MAX_INPUT_TOKEN).sum()}")

# Hapus HANYA entri yang target-nya melebihi 64 token (sangat sedikit)
sebelum_token = len(df)
df = df[df["token_target"] <= MAX_TARGET_TOKEN].reset_index(drop=True)

print(f"\nDihapus karena target >64 token : {sebelum_token - len(df)} entri")
print(f"Dataset final                   : {len(df)} entri")

Melakukan truncation input ke 512 token...


100%|██████████| 7969/7969 [00:05<00:00, 1391.17it/s]


Statistik token INPUT (setelah truncation):
count    7969.0
mean      490.2
std        54.5
min        36.0
25%       508.0
50%       512.0
75%       512.0
max       512.0
Name: token_input, dtype: float64
Entri masih melebihi 512 token: 0

Dihapus karena target >64 token : 43 entri
Dataset final                   : 7926 entri


## 9. Pembagian Dataset (80/10/10)

In [15]:
df_train, df_sisanya = train_test_split(df, test_size=0.20, random_state=42, shuffle=True)
df_val, df_test      = train_test_split(df_sisanya, test_size=0.50, random_state=42, shuffle=True)

print("Split dataset (80/10/10):")
print(f"  Train    : {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)")
print(f"  Validasi : {len(df_val)}  ({len(df_val)/len(df)*100:.1f}%)")
print(f"  Test     : {len(df_test)}  ({len(df_test)/len(df)*100:.1f}%)")

Split dataset (80/10/10):
  Train    : 6340 (80.0%)
  Validasi : 793  (10.0%)
  Test     : 793  (10.0%)


## 10. Simpan Dataset Hasil Preprocessing

In [16]:
FOLDER_OUTPUT = "/kaggle/working/data_preprocessed"
os.makedirs(FOLDER_OUTPUT, exist_ok=True)

KOLOM_SIMPAN = ["input_model", "target_model"]

# Simpan CSV
df_train[KOLOM_SIMPAN].to_csv(f"{FOLDER_OUTPUT}/train.csv", index=False)
df_val[KOLOM_SIMPAN].to_csv(f"{FOLDER_OUTPUT}/val.csv",     index=False)
df_test[KOLOM_SIMPAN].to_csv(f"{FOLDER_OUTPUT}/test.csv",   index=False)
df.to_csv(f"{FOLDER_OUTPUT}/full_preprocessed.csv",         index=False)

# Simpan JSON Lines (format HuggingFace datasets)
def simpan_jsonl(dataframe, path):
    with open(path, "w", encoding="utf-8") as f:
        for _, baris in dataframe.iterrows():
            record = {
                "input_text" : baris["input_model"],
                "target_text": baris["target_model"]
            }
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

simpan_jsonl(df_train, f"{FOLDER_OUTPUT}/train.jsonl")
simpan_jsonl(df_val,   f"{FOLDER_OUTPUT}/val.jsonl")
simpan_jsonl(df_test,  f"{FOLDER_OUTPUT}/test.jsonl")

print(f"Dataset berhasil disimpan di: {FOLDER_OUTPUT}")
print("  train.csv  / train.jsonl")
print("  val.csv    / val.jsonl")
print("  test.csv   / test.jsonl")
print("  full_preprocessed.csv")

Dataset berhasil disimpan di: /kaggle/working/data_preprocessed
  train.csv  / train.jsonl
  val.csv    / val.jsonl
  test.csv   / test.jsonl
  full_preprocessed.csv


## 11. Ringkasan Akhir

In [17]:
print("=" * 60)
print("RINGKASAN PREPROCESSING")
print("=" * 60)
print(f"Dataset awal                        : {jumlah_awal}")
print(f"Setelah hapus missing value         : {sebelum_filter_bahasa}")
print(f"Setelah filter bahasa Indonesia     : {sebelum_filter_bilingual}")
print(f"Setelah hapus bilingual             : {sebelum_filtrasi}")
print(f"Setelah filtrasi panjang & duplikat : {sebelum_token}")
print(f"Dataset final (setelah truncation)  : {len(df)}  ← jauh lebih banyak dari sebelumnya (128)")
print()
print("Distribusi bahasa (dataset final):")
print(df["kategori_bahasa"].value_counts().to_string())
print()
print(f"Rata-rata token input (setelah truncation) : {df['token_input'].mean():.1f}")
print(f"Rata-rata token target                     : {df['token_target'].mean():.1f}")
print(f"Rata-rata jumlah keyphrase per entri       : {df['jumlah_keyphrase_bersih'].mean():.2f}")
print()
print(f"  Train    : {len(df_train)}")
print(f"  Validasi : {len(df_val)}")
print(f"  Test     : {len(df_test)}")
print("=" * 60)

print("\nContoh data siap pakai (train):")
for i in range(min(2, len(df_train))):
    baris = df_train.iloc[i]
    print(f"\n[{i+1}] INPUT  : {baris['input_model'][:150]}...")
    print(f"    TARGET : {baris['target_model']}")

RINGKASAN PREPROCESSING
Dataset awal                        : 9461
Setelah hapus missing value         : 9461
Setelah filter bahasa Indonesia     : 8922
Setelah hapus bilingual             : 7975
Setelah filtrasi panjang & duplikat : 7969
Dataset final (setelah truncation)  : 7926  ← jauh lebih banyak dari sebelumnya (128)

Distribusi bahasa (dataset final):
kategori_bahasa
INDONESIA    7926

Rata-rata token input (setelah truncation) : 490.1
Rata-rata token target                     : 26.6
Rata-rata jumlah keyphrase per entri       : 3.89

  Train    : 6340
  Validasi : 793
  Test     : 793

Contoh data siap pakai (train):

[1] INPUT  : generate keyphrases: masalah utama yang dihadapi oleh para pembudidaya ikan lele dumbo saat ini adalah inefisiensi input produksi yaitu pemanfaatan pa...
    TARGET : catfish, biofloc, technology

[2] INPUT  : generate keyphrases: fasilitas pendukung daya tarik wisata kuliner di seputar cikapundung river spot, kota bandung adalah hasil penelitian dala